# 🐼 Pandas Basics for Exploratory Data Analysis (EDA)

**Goal:** by the end of this notebook you can load a messy dataset, understand it, clean it, summarize it, and plot it using pandas.

**How to use it**
1. Run cells top to bottom (`Shift + Enter`). Later cells depend on earlier ones.
2. Change the code and re-run it. Breaking things is the fastest way to learn.
3. Each section ends with a **✏️ Try it** exercise. Solutions are hidden under "Show solution".

**Roadmap**

| # | Section | Key tools |
|---|---------|-----------|
| 1 | Series & DataFrames | `pd.Series`, `pd.DataFrame` |
| 2 | Loading data | `read_csv` |
| 3 | First look | `head`, `info`, `describe`, `shape` |
| 4 | Selecting & filtering | `[]`, `loc`, `iloc`, boolean masks, `query` |
| 5 | Data quality & cleaning | `isna`, `fillna`, `drop_duplicates`, `str`, `astype` |
| 6 | Creating new columns | arithmetic, `apply`, `map`, `cut`, `dt` |
| 7 | Grouping & summarizing | `groupby`, `agg`, `pivot_table`, `value_counts`, `crosstab` |
| 8 | Relationships & visuals | `corr`, `hist`, `boxplot`, `scatter`, heatmap |
| 9 | Mini EDA project | put it all together |

## 0. Setup

`pandas` handles tabular data, `numpy` handles numbers/arrays, `matplotlib` and `seaborn` draw charts. By convention they are imported as `pd`, `np`, `plt`, `sns`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", 30)   # show more columns when printing
sns.set_theme(style="whitegrid")           # nicer default plots

print("pandas version:", pd.__version__)

## 1. The two core objects

- **Series** = one column (a labeled 1-D array).
- **DataFrame** = a table = a collection of Series sharing the same index (row labels).

Almost everything in pandas is "do something to a column (Series)" or "do something to a table (DataFrame)".

In [ ]:
# A Series: values + an index
ages = pd.Series([23, 35, 41, 29], name="age")
print(ages)
print()
print("Mean age:", ages.mean())
print("Ages over 30:\n", ages[ages > 30])

In [ ]:
# A DataFrame built from a dictionary: keys -> column names, lists -> column values
people = pd.DataFrame({
    "name": ["Ana", "Ben", "Chen", "Dara"],
    "age": [23, 35, 41, 29],
    "city": ["Lima", "Austin", "Taipei", "Lagos"],
})
people

In [ ]:
# Every DataFrame has these basic attributes
print("Shape (rows, cols):", people.shape)
print("Columns:", list(people.columns))
print("Index:", people.index)
print("Data types:\n", people.dtypes)

# A single column is a Series
type(people["age"])

### ✏️ Try it 1
Create a DataFrame called `pets` with columns `name`, `species`, `weight_kg` for 4 pets of your choice. Then print the mean of `weight_kg`.

In [ ]:
# your code here

<details><summary>Show solution</summary>

```python
pets = pd.DataFrame({
    "name": ["Rex", "Tom", "Bubbles", "Polly"],
    "species": ["dog", "cat", "fish", "parrot"],
    "weight_kg": [20.5, 4.2, 0.1, 0.4],
})
print(pets["weight_kg"].mean())
```
</details>

## 2. Loading data

In real projects you load a file: `pd.read_csv("file.csv")`, `pd.read_excel(...)`, `pd.read_json(...)`, or `pd.read_sql(...)`.

To keep this notebook self-contained, the next cell **creates a practice file** called `sales.csv` (an online store's orders, ~500 rows) with realistic problems baked in: missing values, duplicates, inconsistent text, and an outlier. You don't need to understand that cell. Just run it.

In [ ]:
# ---- Generates a messy practice dataset and saves it as sales.csv (no need to study this cell) ----
rng = np.random.default_rng(42)
n = 500

base_price = {"Electronics": 300, "Clothing": 45, "Home": 80, "Books": 18, "Sports": 60}
category = rng.choice(list(base_price), n, p=[.25, .25, .20, .15, .15])

raw = pd.DataFrame({
    "order_id": np.arange(1001, 1001 + n),
    "order_date": (pd.Timestamp("2025-01-01") + pd.to_timedelta(rng.integers(0, 365, n), unit="D")).strftime("%Y-%m-%d"),
    "customer_age": rng.normal(36, 11, n).round().clip(18, 75),
    "gender": rng.choice(["Male", "Female"], n, p=[.48, .52]),
    "region": rng.choice(["North", "South", "East", "West"], n),
    "category": category,
    "unit_price": [round(base_price[c] * rng.lognormal(0, 0.25), 2) for c in category],
    "quantity": rng.integers(1, 6, n),
    "discount": rng.choice([0, 0.05, 0.10, 0.20], n, p=[.5, .2, .2, .1]),
    "rating": rng.choice([1, 2, 3, 4, 5], n, p=[.05, .1, .2, .35, .3]).astype(float),
    "payment_method": rng.choice(["Card", "PayPal", "Cash"], n, p=[.6, .3, .1]),
})

# inject real-world mess
raw.loc[rng.choice(n, 25, replace=False), "customer_age"] = np.nan          # missing ages
raw.loc[rng.choice(n, 40, replace=False), "rating"] = np.nan                # missing ratings
idx = rng.choice(n, 30, replace=False)
raw.loc[idx, "region"] = raw.loc[idx, "region"].str.lower() + " "           # 'north ' instead of 'North'
raw.loc[7, "unit_price"] = 9999.0                                           # an obvious data-entry error
raw = pd.concat([raw, raw.iloc[:8]], ignore_index=True)                     # 8 duplicate rows

raw.to_csv("sales.csv", index=False)
print("Saved sales.csv with", len(raw), "rows")

In [ ]:
# Load it. This is the line you would use with your own files.
df = pd.read_csv("sales.csv")
df.head()

> **Tip:** In Colab you can upload your own CSV via the 📁 sidebar and load it with `pd.read_csv("your_file.csv")`. Useful options: `sep=";"`, `parse_dates=["col"]`, `usecols=[...]`, `nrows=1000`, `na_values=["?", "N/A"]`.

## 3. First look at the data

The first five minutes of any EDA are always the same. Ask:

1. How big is it? (`shape`)
2. What does it look like? (`head`, `tail`, `sample`)
3. What are the columns and types, and what's missing? (`info`)
4. What do the numbers look like? (`describe`)

In [ ]:
print("Shape:", df.shape)       # (rows, columns)
df.head()                       # first 5 rows. Try df.head(10), df.tail(), df.sample(5)

In [ ]:
df.info()

**Reading `info()`:** `Non-Null Count` lower than the number of rows means missing values. `Dtype` shows how pandas stores each column. Notice that `order_date` is `object` (or `str` in newer pandas), meaning plain text, not a real date. We'll fix that soon.

In [ ]:
# Summary statistics for numeric columns
df.describe()

**Reading `describe()`:** compare `mean` vs `50%` (median), and look at `min`/`max`. Look at `unit_price`: the max (9999) is wildly larger than the 75th percentile. That's a red flag we'll investigate.

In [ ]:
# Summary for text (categorical) columns
df.describe(include="object")

In [ ]:
# Counting categories: one of the most-used EDA commands
df["category"].value_counts()

In [ ]:
df["payment_method"].value_counts(normalize=True)   # proportions instead of counts

In [ ]:
# How many unique values? What are they?
print(df["region"].nunique())
print(df["region"].unique())

Notice `region` has **more than 4 unique values** because `'north '` and `'North'` are treated as different. Data quality issue #1.

### ✏️ Try it 2
1. How many rows have `quantity` equal to 5? (hint: `(df["quantity"] == 5).sum()`)
2. What is the most common `rating`?
3. What is the average `customer_age`?

In [ ]:
# your code here

<details><summary>Show solution</summary>

```python
(df["quantity"] == 5).sum()
df["rating"].value_counts().idxmax()    # or df["rating"].mode()[0]
df["customer_age"].mean()               # NaN values are skipped automatically
```
</details>

## 4. Selecting and filtering

| Goal | Syntax |
|------|--------|
| One column | `df["col"]` |
| Several columns | `df[["a", "b"]]` |
| Rows/cols by **label** | `df.loc[rows, cols]` |
| Rows/cols by **position** | `df.iloc[rows, cols]` |
| Rows by condition | `df[condition]` |

In [ ]:
df["unit_price"].head()                 # one column -> Series

In [ ]:
df[["order_id", "category", "unit_price"]].head()   # list of columns -> DataFrame

In [ ]:
# loc: by label (end is INCLUDED).  iloc: by position (end is EXCLUDED)
print(df.loc[0:2, ["order_id", "category"]])
print()
print(df.iloc[0:2, 0:3])

### Boolean filtering (the heart of EDA)
A comparison like `df["unit_price"] > 500` produces a True/False Series (a **mask**). Passing it back into `df[...]` keeps only the `True` rows. Combine conditions with `&` (and), `|` (or), `~` (not), and **always wrap each condition in parentheses**.

In [ ]:
# Expensive items
df[df["unit_price"] > 500].head()

In [ ]:
# Books with a high rating
df[(df["category"] == "Books") & (df["rating"] >= 4)].head()

In [ ]:
# Membership test
df[df["payment_method"].isin(["PayPal", "Cash"])].head()

In [ ]:
# .query() reads like English and is great for quick exploration
df.query("quantity >= 4 and discount > 0").head()

In [ ]:
# Sorting
df.sort_values("unit_price", ascending=False).head()

In [ ]:
# Top-N shortcut
df.nlargest(5, "unit_price")

### ✏️ Try it 3
Find all orders from the `"West"` region (careful: the data is still messy!) paid by `"Card"` with a discount of at least 10%. How many are there?

In [ ]:
# your code here

<details><summary>Show solution</summary>

```python
mask = (df["region"] == "West") & (df["payment_method"] == "Card") & (df["discount"] >= 0.10)
print(mask.sum())
df[mask].head()
```
Note this misses rows where region is `'west '`. After cleaning in Section 5 the count will be higher.
</details>

## 5. Data quality and cleaning

A typical cleaning checklist:

1. Missing values
2. Duplicate rows
3. Wrong data types
4. Inconsistent text
5. Outliers / impossible values

Always **inspect first, then fix**, and keep the raw data untouched by working on a copy.

### 5.1 Missing values

In [ ]:
df.isna().sum()                          # count of missing values per column

In [ ]:
(df.isna().mean() * 100).round(1)        # percent missing per column

**Options for handling missing values**

| Strategy | When it makes sense | Code |
|----------|--------------------|------|
| Drop rows | Few rows affected, or the column is essential | `df.dropna(subset=["col"])` |
| Fill with a constant | Missing has a natural meaning (e.g. 0, "Unknown") | `df["col"].fillna(0)` |
| Fill with median/mean | Numeric column, modest missingness | `df["col"].fillna(df["col"].median())` |
| Leave as NaN | Most pandas stats skip NaN automatically | (do nothing) |

There is no universal right answer. Think about **why** the data is missing.

In [ ]:
clean = df.copy()      # work on a copy so the original stays intact

# Age: fill with the median (robust to outliers)
clean["customer_age"] = clean["customer_age"].fillna(clean["customer_age"].median())

# Rating: missing probably means "not rated", so leave as NaN. mean() and describe() skip NaN.
clean.isna().sum()

### 5.2 Duplicates

In [ ]:
print("Duplicate rows:", clean.duplicated().sum())
clean[clean.duplicated(keep=False)].sort_values("order_id").head(6)   # see them side by side

In [ ]:
clean = clean.drop_duplicates()
print("Rows after dropping duplicates:", len(clean))

### 5.3 Fixing data types
`order_date` was read as text. Converting it to a real datetime unlocks `.dt` features (month, weekday, differences, resampling).

In [ ]:
clean["order_date"] = pd.to_datetime(clean["order_date"])

# Low-cardinality text columns are better as 'category' (saves memory, signals intent)
clean["category"] = clean["category"].astype("category")

clean.dtypes

### 5.4 Inconsistent text
The `.str` accessor applies string methods to a whole column: `.str.strip()`, `.str.lower()`, `.str.title()`, `.str.contains()`, `.str.replace()`.

In [ ]:
print("Before:", sorted(clean["region"].unique()))

clean["region"] = clean["region"].str.strip().str.title()

print("After: ", sorted(clean["region"].unique()))

### 5.5 Outliers
Look at both the numbers and a picture. A **boxplot** marks points beyond 1.5×IQR as outliers.

In [ ]:
clean["unit_price"].describe()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
sns.boxplot(x=clean["unit_price"], ax=axes[0])
axes[0].set_title("unit_price (with the extreme value)")
sns.boxplot(x=clean["unit_price"][clean["unit_price"] < 5000], ax=axes[1])
axes[1].set_title("unit_price (extreme value removed)")
plt.tight_layout()
plt.show()

In [ ]:
# Inspect before deleting anything
clean.nlargest(3, "unit_price")

A 9,999 price for a single order stands far above everything else, including electronics (which cost ~300 on average). We'll treat it as a data-entry error and remove it.

> ⚠️ **Rule of thumb:** an outlier isn't automatically wrong. Investigate it. Remove it only when you have a reason (error, impossible value). Otherwise keep it, or report results with and without it.

You can also flag outliers with the IQR rule:

In [ ]:
q1, q3 = clean["unit_price"].quantile([0.25, 0.75])
iqr = q3 - q1
lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
print(f"IQR bounds: {lower:.1f} to {upper:.1f}")
print("Values flagged:", ((clean["unit_price"] < lower) | (clean["unit_price"] > upper)).sum())
# Many flags are legitimately expensive electronics, not errors. Context matters!

In [ ]:
clean = clean[clean["unit_price"] < 5000].reset_index(drop=True)
print("Final shape:", clean.shape)
clean.info()

### ✏️ Try it 4
Using `clean`, check that `region` now has exactly 4 unique values, and count how many orders each region has.

In [ ]:
# your code here

<details><summary>Show solution</summary>

```python
print(clean["region"].nunique())
clean["region"].value_counts()
```
</details>

## 6. Creating new columns (feature engineering)

New columns are created by assignment: `df["new"] = ...`. Operations are **vectorized**: they apply to every row at once, with no loops.

In [ ]:
df2 = clean.copy()

# Arithmetic between columns
df2["total"] = (df2["unit_price"] * df2["quantity"] * (1 - df2["discount"])).round(2)
df2[["unit_price", "quantity", "discount", "total"]].head()

In [ ]:
# Date parts via the .dt accessor
df2["month"] = df2["order_date"].dt.month
df2["month_name"] = df2["order_date"].dt.month_name()
df2["weekday"] = df2["order_date"].dt.day_name()
df2[["order_date", "month", "month_name", "weekday"]].head()

In [ ]:
# Binning a numeric column into groups with pd.cut
df2["age_group"] = pd.cut(
    df2["customer_age"],
    bins=[17, 25, 35, 50, 100],
    labels=["18-25", "26-35", "36-50", "51+"],
)
df2["age_group"].value_counts().sort_index()

In [ ]:
# np.where: if/else for a whole column
df2["high_value"] = np.where(df2["total"] > 500, "yes", "no")

# .map(): translate values using a dictionary
df2["gender_code"] = df2["gender"].map({"Female": 0, "Male": 1})

# .apply(): run a custom function on each value (flexible, but slower than vectorized code)
def rating_label(r):
    if pd.isna(r):
        return "Not rated"
    return "Happy" if r >= 4 else "Neutral/Unhappy"

df2["sentiment"] = df2["rating"].apply(rating_label)
df2[["total", "high_value", "gender", "gender_code", "rating", "sentiment"]].head()

In [ ]:
# Rename or drop columns
df2 = df2.rename(columns={"customer_age": "age"})
df2 = df2.drop(columns=["gender_code"])
df2.columns.tolist()

### ✏️ Try it 5
Create a column `is_weekend` that is `True` when the order was placed on a Saturday or Sunday. (hint: `df2["order_date"].dt.dayofweek` gives Monday=0 … Sunday=6, and `.isin([5, 6])` is handy.) Then find what fraction of orders happen on weekends.

In [ ]:
# your code here

<details><summary>Show solution</summary>

```python
df2["is_weekend"] = df2["order_date"].dt.dayofweek.isin([5, 6])
df2["is_weekend"].mean()      # mean of booleans = fraction True
```
</details>

## 7. Grouping and summarizing

`groupby` follows **split → apply → combine**: split rows into groups, compute something per group, combine the results.

```
df.groupby("column_to_group_by")["column_to_summarize"].aggregation()
```

In [ ]:
# Revenue per category
df2.groupby("category", observed=True)["total"].sum().sort_values(ascending=False)

In [ ]:
# Several statistics at once with named aggregation
summary = df2.groupby("category", observed=True).agg(
    orders=("order_id", "count"),
    revenue=("total", "sum"),
    avg_order=("total", "mean"),
    avg_rating=("rating", "mean"),
).round(2).sort_values("revenue", ascending=False)
summary

In [ ]:
# Group by two columns
df2.groupby(["region", "payment_method"])["total"].mean().round(2)

### Pivot tables and crosstabs
Think "Excel pivot table": rows × columns, with an aggregated value in each cell.

In [ ]:
pd.pivot_table(
    df2,
    index="region",
    columns="category",
    values="total",
    aggfunc="sum",
    observed=True,
).round(0)

In [ ]:
# crosstab counts combinations; normalize="index" turns each row into percentages
pd.crosstab(df2["category"], df2["payment_method"], normalize="index").round(2)

In [ ]:
# Time trend: group by month
monthly = df2.groupby(df2["order_date"].dt.to_period("M"))["total"].sum()
monthly.plot(marker="o", figsize=(9, 3.5), title="Monthly revenue")
plt.ylabel("Revenue")
plt.show()

### ✏️ Try it 6
1. Which `region` has the highest **average** order `total`?
2. Which weekday has the most orders? (hint: `value_counts()` on `weekday`)

In [ ]:
# your code here

<details><summary>Show solution</summary>

```python
df2.groupby("region")["total"].mean().sort_values(ascending=False)
df2["weekday"].value_counts()
```
</details>

## 8. Relationships and visuals

EDA is mostly about **distributions** (one variable) and **relationships** (two or more variables).

| Question | Chart |
|----------|-------|
| How is a number distributed? | histogram, boxplot |
| How do categories compare? | bar chart |
| Do two numbers move together? | scatter plot, correlation |
| How do distributions differ across groups? | boxplot by group |
| Trend over time? | line chart |

### 8.1 Distributions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))

df2["total"].plot.hist(bins=30, ax=axes[0], title="Order total (skewed right)")
df2["age"].plot.hist(bins=20, ax=axes[1], title="Customer age (roughly bell-shaped)")

plt.tight_layout()
plt.show()

print("Skewness of total:", round(df2["total"].skew(), 2), "(> 0 means a long right tail)")

### 8.2 Categories

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))

df2["category"].value_counts().plot.bar(ax=axes[0], title="Orders per category")
df2.groupby("category", observed=True)["total"].sum().sort_values().plot.barh(ax=axes[1], title="Revenue per category")

plt.tight_layout()
plt.show()

In [ ]:
# Compare distributions across groups
plt.figure(figsize=(8, 4))
sns.boxplot(data=df2, x="category", y="total")
plt.title("Order total by category")
plt.show()

### 8.3 Relationships between numbers
**Correlation** (`corr`) ranges from -1 to +1. Near 0 means no linear relationship. It measures linear association only, and **correlation ≠ causation**.

In [ ]:
num_cols = ["age", "unit_price", "quantity", "discount", "rating", "total"]
corr = df2[num_cols].corr()
corr.round(2)

In [ ]:
plt.figure(figsize=(6.5, 5))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1)
plt.title("Correlation heatmap")
plt.show()

In [ ]:
plt.figure(figsize=(6, 4))
sns.scatterplot(data=df2, x="unit_price", y="total", hue="category", alpha=0.7)
plt.title("Unit price vs order total")
plt.show()

In [ ]:
# A fast overview of many relationships at once
sns.pairplot(df2[["age", "unit_price", "quantity", "total"]], corner=True)
plt.show()

### ✏️ Try it 7
Does the average `rating` differ by `category`? Compute it with `groupby`, then draw a bar chart. Is the difference big enough to matter?

In [ ]:
# your code here

<details><summary>Show solution</summary>

```python
r = df2.groupby("category", observed=True)["rating"].mean().sort_values()
r.plot.barh(title="Average rating by category")
plt.xlim(1, 5)
plt.show()
```
The averages sit in a narrow band (about 3.5 to 3.9), and with only ~100 orders per category that gap is within normal random noise. In this synthetic data ratings were generated independently of category, so there is **no real difference**. Always ask whether a difference is large enough to matter.
</details>

## 9. Mini EDA project

Time to practice the full workflow on your own. A good EDA follows this loop:

> **Look → Clean → Ask a question → Answer with a table or chart → Write the insight in plain words**

### The checklist (reuse it on every dataset)

1. `df.shape`, `df.head()`, `df.info()`, `df.describe()`
2. Missing values: `df.isna().sum()`
3. Duplicates: `df.duplicated().sum()`
4. Data types: dates parsed? categories as categories?
5. Text consistency: `value_counts()` on categorical columns
6. Outliers: boxplots, `describe()`, `nlargest()`
7. Distributions of each key variable
8. Relationships: groupby, pivot, corr, scatter
9. Summarize: 3–5 findings and what you'd investigate next

### Your challenge
Use `df2` (the cleaned data with the new columns) to answer the questions below. Write each insight as a sentence in a markdown cell.

1. What is the total revenue, and which **month** had the highest revenue?
2. Which **category × region** combination generates the most revenue?
3. Do customers who use **PayPal** spend more per order than those who use **Card** or **Cash**?
4. Which **age group** buys the most Electronics (by order count)?
5. What share of revenue comes from the top 10% of orders? (hint: `quantile(0.9)`)

In [ ]:
# your analysis here

<details><summary>Show sample solutions</summary>

```python
# 1
print(df2["total"].sum())
print(df2.groupby("month_name")["total"].sum().idxmax())

# 2
df2.groupby(["category", "region"], observed=True)["total"].sum().nlargest(5)

# 3
df2.groupby("payment_method")["total"].agg(["mean", "median", "count"])

# 4
df2[df2["category"] == "Electronics"]["age_group"].value_counts()

# 5
cut = df2["total"].quantile(0.9)
df2.loc[df2["total"] >= cut, "total"].sum() / df2["total"].sum()
```
</details>

## 📋 Cheat sheet

| Task | Code |
|------|------|
| Load CSV | `pd.read_csv("f.csv", parse_dates=["date"])` |
| Size / types | `df.shape`, `df.dtypes`, `df.info()` |
| Peek | `df.head()`, `df.tail()`, `df.sample(5)` |
| Summary stats | `df.describe()`, `df.describe(include="object")` |
| Count categories | `df["c"].value_counts(normalize=True)` |
| Select | `df["c"]`, `df[["a","b"]]`, `df.loc[...]`, `df.iloc[...]` |
| Filter | `df[(df.a > 5) & (df.b == "x")]`, `df.query("a > 5")` |
| Sort | `df.sort_values("c", ascending=False)`, `df.nlargest(5, "c")` |
| Missing | `df.isna().sum()`, `fillna()`, `dropna()` |
| Duplicates | `df.duplicated().sum()`, `drop_duplicates()` |
| Types | `astype("category")`, `pd.to_datetime()`, `pd.to_numeric(errors="coerce")` |
| Text | `df.c.str.strip().str.lower()` |
| New column | `df["x"] = df.a * df.b`, `np.where`, `.map`, `.apply`, `pd.cut` |
| Dates | `df.d.dt.year`, `.dt.month_name()`, `.dt.day_name()` |
| Group | `df.groupby("c")["v"].agg(["mean","sum","count"])` |
| Pivot | `pd.pivot_table(df, index=, columns=, values=, aggfunc=)` |
| Cross-tab | `pd.crosstab(df.a, df.b, normalize="index")` |
| Correlation | `df.corr(numeric_only=True)` |
| Plots | `df.c.plot.hist()`, `sns.boxplot()`, `sns.scatterplot()`, `sns.heatmap()` |

### Where to go next
- Official docs: *10 minutes to pandas* (pandas.pydata.org)
- Practice on real datasets: Kaggle (Titanic, Housing Prices), `sns.load_dataset("tips")`, `sns.load_dataset("penguins")`
- Next topics: `merge`/`join`, `melt`/`pivot`, `rolling` windows, `ydata-profiling` for automatic EDA reports

Happy exploring! 🐼